# Bài 1: Xây dựng mô hình FastText từ đầu bằng Python (sử dụng PyTorch)

**Mục tiêu:** Xây dựng mô hình phân loại cảm xúc bằng cách sử dụng đặc trưng n-gram ký tự, mô phỏng kiến trúc FastText.
**Luồng xử lý:** Tiền xử lý -> Tạo Subword (n-gram) -> Xây dựng Vocab -> Dataset/Dataloader -> Mô hình FastText -> Huấn luyện -> Đánh giá.

In [ ]:
import re
import torch
import requests
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from collections import Counter
import matplotlib.pyplot as plt

# Cấu hình các tham số huấn luyện
CONFIG = {
    "lr": 0.001,
    "batch_size": 32,
    "embed_dim": 100,
    "n_gram_min": 3,
    "n_gram_max": 6,
    "epochs": 10,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

NUM_CLASSES = 2

In [ ]:
def load_data_from_url(url):
    """Tải dữ liệu từ GitHub."""
    response = requests.get(url)
    lines = response.text.strip().split('\n')
    data = []
    for line in lines:
        parts = line.split(' ', 1)
        if len(parts) == 2:
            label = 1 if parts[0] == '__label__1' else 0
            data.append((parts[1], label))
    return data

train_url = "https://raw.githubusercontent.com/congnghia0609/ntc-scv/master/data/train.txt"
test_url = "https://raw.githubusercontent.com/congnghia0609/ntc-scv/master/data/test.txt"

train_data = load_data_from_url(train_url)
test_data = load_data_from_url(test_url)

In [ ]:
def clean_text(text):
    """Tiền xử lý văn bản cơ bản."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

def get_char_ngrams(word, n_min, n_max):
    """Tạo n-gram ký tự có bọc dấu < >."""
    word = "<" + word + ">"
    ngrams = []
    for n in range(n_min, n_max + 1):
        for i in range(len(word) - n + 1):
            ngrams.append(word[i:i+n])
    return ngrams

In [ ]:
class FastTextModel(nn.Module):
    """Mô hình FastText đơn giản."""
    def __init__(self, vocab_size, embed_dim, output_dim):
        super(FastTextModel, self).__init__()
        # EmbeddingBag tính trung bình các vector trong mỗi câu
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, output_dim)
        
    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

In [ ]:
# Xây dựng từ điển n-gram
all_ngrams = []
for text, _ in train_data:
    words = clean_text(text).split()
    for word in words:
        all_ngrams.extend(get_char_ngrams(word, CONFIG["n_gram_min"], CONFIG["n_gram_max"]))

counts = Counter(all_ngrams)
vocab = {ngram: i + 1 for i, (ngram, count) in enumerate(counts.items()) if count > 1}
vocab['<UNK>'] = 0

def text_to_indices(text):
    indices = []
    for word in clean_text(text).split():
        for ng in get_char_ngrams(word, CONFIG["n_gram_min"], CONFIG["n_gram_max"]):
            indices.append(vocab.get(ng, vocab['<UNK>']))
    return indices

def collate_batch(batch):
    labels, texts, offsets = [], [], [0]
    for (_text, _label) in batch:
        labels.append(_label)
        processed_text = torch.tensor(text_to_indices(_text), dtype=torch.int64)
        texts.append(processed_text)
        offsets.append(processed_text.size(0))
    label_tensor = torch.tensor(labels, dtype=torch.int64)
    offset_tensor = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_tensor = torch.cat(text_list)
    return text_tensor.to(CONFIG["device"]), label_tensor.to(CONFIG["device"]), offset_tensor.to(CONFIG["device"])

# Chỗ này có lỗi nhỏ ở text_list, sửa lại cho đúng
def collate_batch_fixed(batch):
    labels, texts, offsets = [], [], [0]
    for (_text, _label) in batch:
        labels.append(_label)
        idx = torch.tensor(text_to_indices(_text), dtype=torch.int64)
        texts.append(idx)
        offsets.append(idx.size(0))
    return torch.cat(texts).to(CONFIG["device"]), torch.tensor(labels).to(CONFIG["device"]), torch.tensor(offsets[:-1]).cumsum(0).to(CONFIG["device"])

train_loader = DataLoader(train_data, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_batch_fixed)
model = FastTextModel(len(vocab), CONFIG["embed_dim"], NUM_CLASSES).to(CONFIG["device"])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])

history = {'loss': [], 'acc': []}

for epoch in range(CONFIG["epochs"]):
    model.train()
    total_loss, correct = 0, 0
    for texts, labels, offsets in train_loader:
        optimizer.zero_grad()
        outputs = model(texts, offsets)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    
    print(f"Epoch {epoch+1}: Loss {total_loss/len(train_loader):.4f}, Acc {correct/len(train_data):.4f}")

In [ ]:
model.eval()
test_loader = DataLoader(test_data, batch_size=CONFIG["batch_size"], collate_fn=collate_batch_fixed)
correct = 0
with torch.no_grad():
    for texts, labels, offsets in test_loader:
        outputs = model(texts, offsets)
        correct += (outputs.argmax(1) == labels).sum().item()
print(f"Test Accuracy: {correct / len(test_data):.4f}")

In [ ]:
# Vẽ biểu đồ kết quả
plt.plot(history['loss'])
plt.title('Training Loss')
plt.show()